In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("results/combined_run_details.csv")
df["dataset"] = df["dataset"].replace({"sports2": "sports"})
df.head()

In [ ]:
df.value_counts(["dataset", "combine_strategy", "update_strategy", "rerank_strategy", "alpha_mode"])

In [ ]:
df[(df["combine_strategy"] == "linear-comb") &
    (df["dataset"] == "science")].value_counts(["rerank_strategy", "update_strategy"])

In [ ]:
df.groupby(["combine_strategy", "rerank_strategy", "alpha_mode"], as_index=False)["post_priming_score"].mean()

In [ ]:
subset_df = df[
    (
        (df["combine_strategy"] == "linear-comb") &
        (df["rerank_strategy"] == "none") &
        (df["alpha_mode"] == "sliding")
    )
    |
    (
        (df["combine_strategy"] == "spherical-comb") &
        (df["rerank_strategy"] == "none") &
        (df["alpha_mode"] == "sliding")
    )
    |
    (
        (df["combine_strategy"] == "query-only") &
        (df["rerank_strategy"] == "none") &
        (df["alpha_mode"] == "none")
    )
]

In [ ]:
subset_df.groupby(["dataset", "combine_strategy"])["post_priming_score"].mean()

In [ ]:
import matplotlib.pyplot as plt

NORMALIZATION_DENOM = 11.948459118879
STRATEGY_ORDER = ["query-only", "linear-comb", "spherical-comb"]

# Work on a copy to avoid chained-assignment issues.
analysis_df = subset_df.copy()
analysis_df["post_priming_score_normalized"] = (
    analysis_df["post_priming_score"] / NORMALIZATION_DENOM
)

# Mean normalized score by dataset and combine strategy.
norm_means = (
    analysis_df
    .groupby(["dataset", "combine_strategy"], as_index=False)["post_priming_score_normalized"]
    .mean()
)

# Build an improvement table relative to query-only within each dataset.
baseline = (
    norm_means[norm_means["combine_strategy"] == "query-only"]
    .rename(columns={"post_priming_score_normalized": "query_only_normalized"})
    [["dataset", "query_only_normalized"]]
)

improvement_table = (
    norm_means.merge(baseline, on="dataset", how="left")
    .assign(
        absolute_improvement=lambda d: (
            d["post_priming_score_normalized"] - d["query_only_normalized"]
        ),
        percent_improvement=lambda d: (
            d["absolute_improvement"] / d["query_only_normalized"] * 100
        ),
    )
)

# Keep table rows in requested strategy order.
improvement_table["combine_strategy"] = pd.Categorical(
    improvement_table["combine_strategy"],
    categories=STRATEGY_ORDER,
    ordered=True,
)
improvement_table = improvement_table.sort_values(["dataset", "combine_strategy"])

display(
    improvement_table[[
        "dataset",
        "combine_strategy",
        "post_priming_score_normalized",
        "query_only_normalized",
        "absolute_improvement",
        "percent_improvement",
    ]].round(4)
)

# Plot 1: normalized post-priming score by strategy (includes baseline).
pivot_norm = (
    norm_means.pivot(
        index="dataset",
        columns="combine_strategy",
        values="post_priming_score_normalized",
    )
    .reindex(columns=STRATEGY_ORDER)
)

ax = pivot_norm.plot(kind="bar", figsize=(10, 5))
ax.set_title("Normalized Post-Priming Score by Combine Strategy")
ax.set_xlabel("Dataset")
ax.set_ylabel("Normalized Post-Priming Score")
plt.xticks(rotation=0)
plt.legend(title="Combine Strategy")
plt.tight_layout()
plt.show()

In [ ]:
subset_df2 = df[
    (
        (df["combine_strategy"] == "linear-comb") &
        (df["alpha_mode"] == "sliding")
    )
    |
    (
        (df["combine_strategy"] == "spherical-comb") &
        (df["alpha_mode"] == "sliding")
    )
]

In [ ]:
import matplotlib.pyplot as plt

NORMALIZATION_DENOM = 11.948459118879

plot_df2 = subset_df2.copy()
plot_df2["post_priming_score_normalized"] = (
    plot_df2["post_priming_score"] / NORMALIZATION_DENOM
)

agg = (
    plot_df2
    .groupby(["dataset", "combine_strategy", "rerank_strategy"], as_index=False)["post_priming_score_normalized"]
    .mean()
)

linear_pivot = (
    agg[agg["combine_strategy"] == "linear-comb"]
    .pivot(index="dataset", columns="rerank_strategy", values="post_priming_score_normalized")
    .reindex(columns=["none", "cross-encoder"])
)

spherical_pivot = (
    agg[agg["combine_strategy"] == "spherical-comb"]
    .pivot(index="dataset", columns="rerank_strategy", values="post_priming_score_normalized")
    .reindex(columns=["none", "cross-encoder"])
)

# Optional: keep tables for exact values.
display(linear_pivot.round(4))
display(spherical_pivot.round(4))

# Single figure with two vertically stacked subplots.
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 9), sharex=True)

linear_pivot.plot(kind="bar", ax=axes[0])
axes[0].set_title("Linear-Comb: Normalized Post-Priming Score")
axes[0].set_ylabel("Normalized Score")
axes[0].legend(["no cross-encoder", "with cross-encoder"], title="Rerank Strategy")
axes[0].tick_params(axis="x", rotation=0)

spherical_pivot.plot(kind="bar", ax=axes[1])
axes[1].set_title("Spherical-Comb: Normalized Post-Priming Score")
axes[1].set_xlabel("Dataset")
axes[1].set_ylabel("Normalized Score")
axes[1].legend(["no cross-encoder", "with cross-encoder"], title="Rerank Strategy")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
subset_df3 = df[
    (
        (df["combine_strategy"] == "linear-comb")
    )
    |
    (
        (df["combine_strategy"] == "spherical-comb")
    )
]

In [ ]:
import matplotlib.pyplot as plt

NORMALIZATION_DENOM = 11.948459118879

plot_df3 = subset_df3.copy()
plot_df3["post_priming_score_normalized"] = (
    plot_df3["post_priming_score"] / NORMALIZATION_DENOM
)

# Build one plotting label that supports both fixed alpha values and sliding mode.
plot_df3["alpha_setting"] = plot_df3["alpha_value"].astype(str)
plot_df3.loc[plot_df3["alpha_mode"] == "sliding", "alpha_setting"] = "sliding"

alpha_means = (
    plot_df3
    .groupby(["dataset", "alpha_setting"], as_index=False)["post_priming_score_normalized"]
    .mean()
)

# Order numeric alpha bars ascending, then include sliding at the end if present.
numeric_settings = sorted(
    [s for s in alpha_means["alpha_setting"].unique() if s != "sliding"],
    key=float,
)
ordered_settings = numeric_settings + (["sliding"] if "sliding" in alpha_means["alpha_setting"].unique() else [])

pivot_alpha = (
    alpha_means
    .pivot(index="dataset", columns="alpha_setting", values="post_priming_score_normalized")
    .reindex(columns=ordered_settings)
)

display(pivot_alpha.round(4))

ax = pivot_alpha.plot(kind="bar", figsize=(11, 5))
ax.set_title("Normalized Post-Priming Score by Dataset and Alpha Setting")
ax.set_xlabel("Dataset")
ax.set_ylabel("Normalized Score")
plt.xticks(rotation=0)
plt.legend(title="Alpha Setting")
plt.tight_layout()
plt.show()